Step 1: Set up connection string and file path    

In [9]:
import pandas as pd
from pathlib import Path
import psycopg2
from psycopg2.extras import execute_batch
from urllib.parse import quote_plus

# Configuration for Supabase connection
SUPABASE_HOST = "aws-0-ap-southeast-1.pooler.supabase.com"
SUPABASE_PORT = "6543"
SUPABASE_DB = "postgres"
SUPABASE_USER = "postgres.laremattjpwkgpmwitzv"
SUPABASE_PASSWORD = "Wombat2025@@!!"

# Paths
FINAL_DATA_FOLDER = "../../../pre_data"  
TABLE_NAME = "pre_data"

DATABASE_URL = f"postgresql://{SUPABASE_USER}:{SUPABASE_PASSWORD}@{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}"



Step 2: Column mapping to match db column names

In [17]:
COLUMN_MAPPING = {
    # CSV Column Name : Database Column Name
    'Effective Date': 'effective_date',
    'Fund Name': 'fund_name',
    'Option Name': 'option_name', 
    'Asset Class Name': 'asset_class_name',
    'Int/Ext': 'int_ext',
    'Name/Kind of Investment Item': 'investment_item_name',
    'Currency': 'currency',
    'Stock ID': 'stock_id',
    'Listed Country': 'listed_country',
    'Units Held': 'units_held',
    '% Ownership': 'ownership_percentage',
    'Address': 'address',
    'Value (AUD)': 'value_aud',
    'Weighting': 'weighting'
}


# Step 3: Upload function

In [ ]:
def upload_csvs():
    print("Starting upload...")

    try:
        conn = psycopg2.connect(
            host=SUPABASE_HOST,
            port=SUPABASE_PORT,
            dbname=SUPABASE_DB,
            user=SUPABASE_USER,
            password=SUPABASE_PASSWORD,
            sslmode="require"
        )
        cursor = conn.cursor()

        data_folder = Path(FINAL_DATA_FOLDER)
        csv_files = list(data_folder.glob("*.csv"))
        print(f"Found {len(csv_files)} CSV files in {data_folder}")

        total_rows = 0
        for i, csv_file in enumerate(csv_files, 1):
            print(f"\n Processing {i}/{len(csv_files)}: {csv_file.name}")

            # Load CSV
            df = pd.read_csv(csv_file)
            print(f"   Read {len(df)} rows")

            # Apply column mapping
            df = df.rename(columns=COLUMN_MAPPING)

            # Keep only mapped columns (in case CSV has extras)
            df = df[list(COLUMN_MAPPING.values())]
            print("    Applied column mapping")

            if df.empty:
                print("    Skipped empty file")
                continue
            
            # Data type conversions
            if "effective_date" in df.columns:
                df["effective_date"] = pd.to_datetime(
                    df["effective_date"],
                    format="mixed",
                    dayfirst=True, 
                    errors="coerce"
                )
                df["effective_date"] = df["effective_date"].dt.strftime("%Y-%m-%d")
    
            # Columns that should be numeric
            numeric_cols = ["units_held", "ownership_percentage", "value_aud", "weighting"]

            for col in numeric_cols:
                if col in df.columns:
                    df[col] = (
                        df[col]
                        .astype(str)
                        .str.replace(",", "", regex=False)   # remove commas
                        .str.replace("%", "", regex=False)   # remove percent signs
                        .str.strip()
                    )
                    df[col] = pd.to_numeric(df[col], errors="coerce")

            # Replace NaN with None so psycopg2 inserts NULL
            df = df.where(pd.notnull(df), None)


            # Prepare insert statement
            cols = ', '.join([f'"{c}"' for c in df.columns])
            placeholders = ', '.join(['%s'] * len(df.columns))
            insert_sql = f'INSERT INTO {TABLE_NAME} ({cols}) VALUES ({placeholders})'

            # Execute batch insert
            records = df.to_records(index=False).tolist()
            execute_batch(cursor, insert_sql, records, page_size=1000)

            print(f"    Inserted {len(df)} rows")
            total_rows += len(df)

        conn.commit()
        cursor.close()
        conn.close()

        print("\n=============================")
        print(f" Upload complete! {total_rows} rows inserted into {TABLE_NAME}")
        print("=============================")

    except Exception as e:
        print(f" Upload failed: {e}")


# ==============================
# RUN SCRIPT
# ==============================
if __name__ == "__main__":
    upload_csvs()


Starting upload...
📂 Found 12 CSV files in ..\..\..\pre_data

➡️ Processing 1/12: art_pre_cleaned.csv
   Read 5493 rows
   ✅ Applied column mapping
   ✅ Inserted 5493 rows

➡️ Processing 2/12: aussie_pre_cleaned.csv
   Read 4130 rows
   ✅ Applied column mapping
   ✅ Inserted 4130 rows

➡️ Processing 3/12: aware_pre_cleaned.csv
   Read 2613 rows
   ✅ Applied column mapping
   ✅ Inserted 2613 rows

➡️ Processing 4/12: care_pre_cleaned.csv
   Read 2295 rows
   ✅ Applied column mapping
   ✅ Inserted 2295 rows

➡️ Processing 5/12: cbus_pre_cleaned.csv
   Read 2176 rows
   ✅ Applied column mapping
   ✅ Inserted 2176 rows

➡️ Processing 6/12: equip_pre_cleaned.csv
   Read 1964 rows
   ✅ Applied column mapping
   ✅ Inserted 1964 rows

➡️ Processing 7/12: hesta_pre_cleaned.csv
   Read 3136 rows
   ✅ Applied column mapping
   ✅ Inserted 3136 rows

➡️ Processing 8/12: hostplus_pre_cleaned.csv
   Read 2504 rows
   ✅ Applied column mapping
   ✅ Inserted 2504 rows

➡️ Processing 9/12: ngs_pre_cleane